# Torneo de Clasificadores – Dataset Wearable (Sedentarismo)

**Proyecto:** Wearable: Minería de Datos en Sensores de Sedentarismo  
**Objetivo:** Se hizo la comparacion entre Arbol de Decisión, Naive Bayes y KNN sobre el mismo dataset.

Variable objetivo: `Nivel_Sedentarismo` (Alto/Bajo).  
Features (sin fuga de datos): `VeryActiveMinutes`, `LightlyActiveMinutes`, `FairlyActiveMinutes`, `Calories`, `Nivel_Pasos` (one-hot).  
Partición: `train_test_split(test_size=0.25, random_state=42, stratify=y)`.

In [4]:
# ============================================================
# Torneo de clasificadores sobre el dataset wearable (sedentarismo)
# CORREGIDO: sin fuga de datos (sin SedentaryMinutes ni TotalSteps)
# ============================================================

import time
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

# ---------- 1. Cargar dataset ----------
RUTA = Path("../Datacrude/02IntermediateData/Sensores_limpio.csv")
df = pd.read_csv(RUTA)

# ---------- 2. Definir X e y ----------
# Variable objetivo: Nivel_Sedentarismo (Alto/Bajo) -> binaria
y = df["Nivel_Sedentarismo"].map({"Alto Sedentarismo": 1, "Bajo Sedentarismo": 0})

# Features SIN leakage: se excluyen SedentaryMinutes y TotalSteps
# (son las variables que definen la etiqueta)
num = ["VeryActiveMinutes", "LightlyActiveMinutes",
       "FairlyActiveMinutes", "Calories"]
X = pd.get_dummies(df[num + ["Nivel_Pasos"]], columns=["Nivel_Pasos"])

# ---------- 3. Partición única (misma para todos) ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

# ---------- 4. Los 3 rivales ----------
modelos = {
    "Arbol (prof=4)": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Naive Bayes": GaussianNB(),
    "KNN (k=5)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
}

# ---------- 5. Torneo: accuracy train/test + tiempo de entrenamiento ----------
print(f"\n{'modelo':<16} {'entrena(s)':>10} {'acc_train':>10} {'acc_test':>9}")
for nombre, modelo in modelos.items():
    inicio = time.perf_counter()
    modelo.fit(X_train, y_train)
    duracion = time.perf_counter() - inicio
    acc_train = modelo.score(X_train, y_train)
    acc_test = modelo.score(X_test, y_test)
    print(f"{nombre:<16} {duracion:>10.4f} {acc_train:>10.3f} {acc_test:>9.3f}")

# ---------- 6. KNN con y sin escalar: por qué la escala importa ----------
knn_crudo = KNeighborsClassifier(n_neighbors=5)
knn_crudo.fit(X_train, y_train)
print(f"\nKNN k=5 SIN escalar -> accuracy test: {knn_crudo.score(X_test, y_test):.3f}")
print("(sin escalar, las variables de mayor magnitud dominan la distancia euclidiana)")

# ---------- 7. Efecto del valor de k ----------
escalador_knn = make_pipeline(StandardScaler(), KNeighborsClassifier())
print("\n   k   acc_train   acc_test")
for k in [1, 3, 5, 11, 31, 101]:
    escalador_knn.set_params(kneighborsclassifier__n_neighbors=k)
    escalador_knn.fit(X_train, y_train)
    print(f"{k:>4}   {escalador_knn.score(X_train, y_train):>9.3f}   "
          f"{escalador_knn.score(X_test, y_test):>8.3f}")

# ---------- 8. (Opcional) KNN con pesos por distancia ----------
knn_dist = make_pipeline(StandardScaler(), KNeighborsClassifier(weights="distance"))
knn_dist.fit(X_train, y_train)
print(f"\nKNN (k=5, weights='distance') -> accuracy test: {knn_dist.score(X_test, y_test):.3f}")

Train: 651 | Test: 217

modelo           entrena(s)  acc_train  acc_test
Arbol (prof=4)       0.0025      0.842     0.816
Naive Bayes          0.0014      0.779     0.774
KNN (k=5)            0.0026      0.856     0.825

KNN k=5 SIN escalar -> accuracy test: 0.811
(sin escalar, las variables de mayor magnitud dominan la distancia euclidiana)

   k   acc_train   acc_test
   1       1.000      0.779
   3       0.871      0.820
   5       0.856      0.825
  11       0.828      0.788
  31       0.797      0.793
 101       0.796      0.793

KNN (k=5, weights='distance') -> accuracy test: 0.816


## 1. Árbol de Decisión (max_depth=4)

**Qué se usó:** `DecisionTreeClassifier(max_depth=4, random_state=42)` sobre las features anteriores.

**Resultados:**
- Accuracy train: **0.842**
- Accuracy test: **0.816**
- Tiempo de entrenamiento: 0.0025 s

**Veredicto:** Accuracy moderado (0.816) con cierto sobreajuste (brecha train/test de 0.026). Su principal ventaja es la **interpretabilidad**: genera reglas legibles (ej. "si VeryActiveMinutes bajo y Calories bajo → riesgo alto"), ideales para explicar decisiones a médicos o pacientes. Recomendado si el negocio exige transparencia.

## 2. Naive Bayes (GaussianNB)

**Qué se usó:** `GaussianNB()` (variante para variables numéricas, asume distribución gaussiana por clase).

**Resultados:**
- Accuracy train: **0.779**
- Accuracy test: **0.774**
- Tiempo de entrenamiento: 0.0014 s (el más rápido)

**Veredicto:** El accuracy más bajo (0.774), pero el más **estable** (brecha train/test de solo 0.005) y el más rápido. Funciona como **línea base honesta**: si un modelo más complejo no supera a Naive Bayes, sobra complejidad. No ofrece reglas interpretables.

## 3. K-Vecinos (KNN, k=5)

**Qué se usó:** `make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))`.

**Resultados:**
- Accuracy train: **0.856**
- Accuracy test: **0.825** (el mejor de los 3)
- Tiempo de entrenamiento: 0.0026 s
- KNN **sin escalar**: accuracy test **0.811** (baja 0.014) → confirma que la escala importa.

**Efecto de k:**
| k | acc_train | acc_test | Diagnóstico |
|---|---|---|---|
| 1 | 1.000 | 0.779 | Sobreajuste (memoriza) |
| 5 | 0.856 | 0.825 | Buen equilibrio |
| 11 | 0.828 | 0.788 | Empeora en test |
| 31 | 0.797 | 0.793 | Subajuste |
| 101 | 0.796 | 0.793 | Subajuste (tiende a la clase mayoritaria) |

**KNN con `weights='distance'`:** accuracy test 0.816 (no mejora respecto al uniforme).

**Veredicto:** Mejor accuracy en test (0.825) con k=5 y escalado. Sin embargo, es **lento en predicción** (busca vecinos en toda la historia) y no es interpretable. Recomendado si solo importa la precisión.

## Conclusión: ¿qué modelo usar para el proyecto?

| Criterio | Árbol (prof=4) | Naive Bayes | KNN (k=5, escalado) |
|---|---|---|---|
| Accuracy test | 0.816 | 0.774 | **0.825** |
| Interpretabilidad | **Alta** (reglas legibles) | Baja | Baja-media |
| Velocidad predicción | Rápida | Instantánea | Lenta (busca vecinos) |
| ¿Requiere escalar? | No | No | Sí (crítico) |
| Riesgo sobreajuste | Moderado (brecha 0.026) | Mínimo | Moderado (brecha 0.031) |

**Recomendación final:**
- Si el negocio exige **explicar decisiones** (alertas a médicos/pacientes, regulación): elegir el **Árbol** (0.816), con reglas legibles.
- Si solo importa la **precisión** y no hay restricción de explicabilidad: elegir **KNN k=5** (0.825), asumiendo el costo de predicción.
- Usar **Naive Bayes (0.774)** como **línea base permanente** para detectar sobrecomplejidad.

> La decisión final la definen la **interpretabilidad y la operación**, no solo el accuracy.